### Homework
Create a new notebook or make a copy of this notebook.
1. Try another categories of Brown Corpus
2. Try changing from first 1000 to first 5000 or randomly at any points (min 1000)
3. Try removing stopwords in Normalization steps
4. Observe the differences in text generation and next word prediction
5. **Go to Canvas:** Upload your notebook, and write your observation in ***N-gram Languge Model exercise***

# 1. Choose text corpus

In [11]:
import nltk
import random
import collections
from nltk.corpus import brown
from nltk.corpus import stopwords

# 2. Change from first 1000 to first 5000

In [12]:
print(len(brown.sents(categories='mystery')))

sentences = brown.sents(categories='mystery')[:5000]
print(f"Number of sentences: {len(sentences)}")
print(f"First sentence: {sentences[1]}")

3886
Number of sentences: 3886
First sentence: ['An', 'interne', ',', 'a', 'nurse', 'and', 'two', 'attendants', 'were', 'in', 'charge', 'of', 'us', '.']


# 2. Pre-processing

In [13]:
# 3. Pre-processing function
def preprocess_text(sentences, remove_stopwords=False):
    processed_sentences = []
    stop_words = set(stopwords.words('english')) if remove_stopwords else set()

    for sentence in sentences:
        # Lowercase and filter alphabetic tokens
        words = [word.lower() for word in sentence if word.isalpha()]
        
        # Filter out stopwords if enabled
        if remove_stopwords:
            words = [word for word in words if word not in stop_words]
            
        # Add sequence markers
        processed = ['<START>'] + words + ['<END>']
        processed_sentences.append(processed)
        
    return processed_sentences

processed_with_stopwords = preprocess_text(sentences, remove_stopwords=False)
processed_without_stopwords = preprocess_text(sentences, remove_stopwords=True)


# 4. Text Generation Vs Text Prediction

## 4.1 Build Bigram Model

In [14]:
def build_bigram_counts(processed_data):
    all_bigrams = []
    for sentence in processed_data:
        for i in range(len(sentence) - 1):
            all_bigrams.append((sentence[i], sentence[i + 1]))
    
    # Calculate conditional frequency distribution for prediction/generation
    cfd = nltk.ConditionalFreqDist(all_bigrams)
    # Also return a flat list frequency distribution for the top bigrams printout
    fd = nltk.FreqDist(all_bigrams)
    return cfd, fd

cfd_kept, fd_kept = build_bigram_counts(processed_with_stopwords)
cfd_removed, fd_removed = build_bigram_counts(processed_without_stopwords)

In [15]:
# Compare Top Bigrams
print("\n--- TOP 5 BIGRAMS WITH STOPWORDS KEPT ---")
for bigram, count in fd_kept.most_common(5):
    print(f"{bigram}: {count}")

print("\n--- TOP 5 BIGRAMS WITH STOPWORDS REMOVED ---")
for bigram, count in fd_removed.most_common(5):
    print(f"{bigram}: {count}")


--- TOP 5 BIGRAMS WITH STOPWORDS KEPT ---
('<START>', 'he'): 417
('<START>', 'the'): 261
('<START>', 'i'): 249
('of', 'the'): 223
('in', 'the'): 216

--- TOP 5 BIGRAMS WITH STOPWORDS REMOVED ---
('<START>', '<END>'): 88
('said', '<END>'): 52
('<START>', 'would'): 39
('<START>', 'could'): 36
('<START>', 'got'): 34


### Text Generation

In [16]:
# Text Generation Function
def generate_sentence(cfd, max_length=15):
    word = '<START>'
    sentence = [word]
    
    while word != '<END>' and len(sentence) < max_length:
        # Pick the next word based on its bigram probability distribution
        if word in cfd and cfd[word]:
            choices = list(cfd[word].keys())
            weights = list(cfd[word].values())
            word = random.choices(choices, weights=weights)[0]
            sentence.append(word)
        else:
            break
    return sentence

### Word prediction

In [17]:
def predict_next_word(cfd, current_word):
    if current_word in cfd and cfd[current_word]:
        return cfd[current_word].max() # Returns the most likely next word
    return "None (Out of vocabulary)"

### The demonstration

In [18]:
# Demonstrate Output Differences
print("\n--- TEXT GENERATION COMPARISON ---")
print("With Stopwords: ", " ".join(generate_sentence(cfd_kept)))
print("Without Stopwords:", " ".join(generate_sentence(cfd_removed)))

print("\n--- NEXT WORD PREDICTION COMPARISON ---")
test_word = "he"
print(f"Predicting next word after '{test_word}' (With Stopwords):", predict_next_word(cfd_kept, test_word))
print(f"Predicting next word after '{test_word}' (Without Stopwords):", predict_next_word(cfd_removed, test_word))


--- TEXT GENERATION COMPARISON ---
With Stopwords:  <START> he drove east toward the two millions ten decks for the peeping tom place
Without Stopwords: <START> hit twice long box long satiric <END>

--- NEXT WORD PREDICTION COMPARISON ---
Predicting next word after 'he' (With Stopwords): was
Predicting next word after 'he' (Without Stopwords): None (Out of vocabulary)
